In [36]:
import time
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import joblib
import torch
import torch.nn as nn
from pathlib import Path
from collections import deque, Counter

from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python.vision import PoseLandmarker, PoseLandmarkerOptions, RunningMode

In [37]:
WINDOW_SIZE    = 45
P2_WINDOW_SIZE = 90    # collect 90 real frames; pad to MAX_SEQUENCE_LENGTH before inference
MAX_SEQUENCE_LENGTH = 120  # must match MAX_SEQUENCE_LENGTH in notebook 06
EMA_ALPHA      = 0.3  # EMA weight for each new prediction (0=frozen, 1=no smoothing)
WEBCAM_ID      = 0

# ── Label maps ───────────────────────────────────────────────
EXERCISE_LABELS = {
    0: "Pull-up",
    1: "Push-up",
    2: "Russian Twist",
    3: "Squat",
}
CORRECTNESS_LABELS = {0: "Incorrect", 1: "Correct"}
CORRECTNESS_COLORS = {0: (0, 0, 220),  1: (0, 220, 0)}

# ── Phase 1 paths ─────────────────────────────────────────────
MODEL_PATH           = Path("../data/models/pose_landmarker_lite.task")
SCALER_PATH          = Path("../data/models/scaler.pkl")
EXERCISE_SVM_PATH    = Path("../data/models/svm_exercise_tuned.pkl")
CORRECTNESS_SVM_PATH = Path("../data/models/svm_correctness_tuned.pkl")
EXERCISE_RF_PATH     = Path("../data/models/rf_exercise_tuned.pkl")
CORRECTNESS_RF_PATH  = Path("../data/models/rf_correctness_tuned.pkl")

# ── Phase 2 paths ─────────────────────────────────────────────
LSTM_PATH = Path("../data/models/LSTM.pt")
TCN_PATH  = Path("../data/models/TCN.pt")

# ── DL constants ──────────────────────────────────────────────
N_FEATURES            = 26
N_EXERCISE_CLASSES    = 4
N_CORRECTNESS_CLASSES = 2
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

NON_FEATURE_COLS = [
    "video_id", "frame_number",
    "exercise_name_encoded", "exercise_correctness_encoded",
]

# ── Verify all files exist ────────────────────────────────────
for p in [MODEL_PATH, SCALER_PATH,
          EXERCISE_SVM_PATH, CORRECTNESS_SVM_PATH,
          EXERCISE_RF_PATH,  CORRECTNESS_RF_PATH,
          LSTM_PATH, TCN_PATH]:
    assert p.exists(), f"❌ File not found: {p}"

print("✅ Config ready.")
print(f"   Rolling window (P1) : {WINDOW_SIZE} frames")
print(f"   Rolling window (P2) : {P2_WINDOW_SIZE} frames")
print(f"   Webcam ID           : {WEBCAM_ID}")
print(f"   Device (DL)         : {DEVICE}")


✅ Config ready.
   Rolling window (P1) : 45 frames
   Rolling window (P2) : 90 frames
   Webcam ID           : 0
   Device (DL)         : cpu


In [38]:
import sys
sys.path.insert(0, "..")
from src.shared_utils import calculate_angle, LANDMARKS, get_coords, get_z, extract_angles

# Sanity check
test_angle = calculate_angle((0, 0), (1, 0), (1, 1))
print(f"Shared utilities loaded. Sanity check (expect 90): {test_angle}°")
print(f"   extract_angles ready. Features per frame: 13 angles + 13 velocities = 26.")


Shared utilities loaded. Sanity check (expect 90): 90.0°
   extract_angles ready. Features per frame: 13 angles + 13 velocities = 26.


In [39]:
# LANDMARKS, get_coords, get_z, extract_angles are imported from src/shared_utils.py above.


In [40]:
POSE_CONNECTIONS = [
    (11, 12),
    (11, 13), (13, 15),
    (12, 14), (14, 16),
    (11, 23), (12, 24),
    (23, 24),
    (23, 25), (25, 27), (27, 29),
    (24, 26), (26, 28), (28, 30),
]

def draw_skeleton(frame, landmarks, h, w):
    for start_idx, end_idx in POSE_CONNECTIONS:
        if start_idx >= len(landmarks) or end_idx >= len(landmarks):
            continue
        x1 = int(landmarks[start_idx].x * w)
        y1 = int(landmarks[start_idx].y * h)
        x2 = int(landmarks[end_idx].x * w)
        y2 = int(landmarks[end_idx].y * h)
        cv2.line(frame, (x1, y1), (x2, y2), (200, 200, 200), 2)

    for name, idx in LANDMARKS.items():
        if idx >= len(landmarks):
            continue
        x = int(landmarks[idx].x * w)
        y = int(landmarks[idx].y * h)
        cv2.circle(frame, (x, y), 5, (0, 255, 0), -1)

print("✅ draw_skeleton ready.")

✅ draw_skeleton ready.


In [41]:
# ── Phase 1: scaler + RF + SVM ────────────────────────────────
scaler          = joblib.load(SCALER_PATH)
exercise_svm    = joblib.load(EXERCISE_SVM_PATH)
correctness_svm = joblib.load(CORRECTNESS_SVM_PATH)
exercise_rf     = joblib.load(EXERCISE_RF_PATH)
correctness_rf  = joblib.load(CORRECTNESS_RF_PATH)

print("✅ Phase 1 models loaded.")
print(f"   Scaler expects {scaler.n_features_in_} features.")

# ── Phase 2: model class definitions ─────────────────────────
import torch.nn.functional as F


class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, encoder_outputs):
        weights = self.attention(encoder_outputs).squeeze(-1)
        weights = F.softmax(weights, dim=1)
        return torch.sum(weights.unsqueeze(-1) * encoder_outputs, dim=1)


class ExerciseLSTM(nn.Module):
    def __init__(self, input_size=N_FEATURES, n_exercise=N_EXERCISE_CLASSES,
                 n_correctness=N_CORRECTNESS_CLASSES,
                 hidden_size=64, num_layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size, hidden_size=hidden_size,
            num_layers=num_layers, batch_first=True, bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.attention = Attention(hidden_size * 2)  # !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
        self.fc_shared = nn.Sequential(
            nn.Linear(hidden_size * 2, 64), nn.ReLU(), nn.Dropout(dropout),  # !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
        )
        self.head_exercise     = nn.Linear(64, n_exercise)
        # Soft MoE: one correctness head per exercise, mixed by exercise probabilities.
        self.heads_correctness = nn.ModuleList(
            [nn.Linear(64, n_correctness) for _ in range(n_exercise)]
        )

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        context     = self.attention(lstm_out)
        shared      = self.fc_shared(context)
        ex_logits   = self.head_exercise(shared)
        mix         = F.softmax(ex_logits, dim=1)
        cor_logits  = sum(
            mix[:, i].unsqueeze(1) * head(shared)
            for i, head in enumerate(self.heads_correctness)
        )
        return ex_logits, cor_logits


class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super().__init__()
        self.chomp_size = chomp_size

    def forward(self, x):
        return x[:, :, :-self.chomp_size].contiguous()


class TemporalBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation, dropout=0.2):
        super().__init__()
        padding       = (kernel_size - 1) * dilation
        self.conv1    = nn.utils.weight_norm(
            nn.Conv1d(in_channels, out_channels, kernel_size,
                      padding=padding, dilation=dilation)
        )
        self.chomp1   = Chomp1d(padding)
        self.relu1    = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)
        self.conv2    = nn.utils.weight_norm(
            nn.Conv1d(out_channels, out_channels, kernel_size,
                      padding=padding, dilation=dilation)
        )
        self.chomp2   = Chomp1d(padding)
        self.relu2    = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)
        self.net      = nn.Sequential(
            self.conv1, self.chomp1, self.relu1, self.dropout1,
            self.conv2, self.chomp2, self.relu2, self.dropout2,
        )
        self.residual_proj = (
            nn.Conv1d(in_channels, out_channels, kernel_size=1)
            if in_channels != out_channels else None
        )
        self.final_relu = nn.ReLU()
        nn.init.normal_(self.conv1.weight, 0.0, 0.01)
        nn.init.normal_(self.conv2.weight, 0.0, 0.01)

    def forward(self, x):
        out = self.net(x)
        res = x if self.residual_proj is None else self.residual_proj(x)
        return self.final_relu(out + res)


class ExerciseTCN(nn.Module):
    def __init__(self, input_size=N_FEATURES, channels=None, kernel_size=3,
                 dilations=None, dropout=0.2):
        super().__init__()
        channels  = channels  or [32, 64, 64]
        dilations = dilations or [1, 2, 4]
        blocks, in_ch = [], input_size
        for out_ch, dil in zip(channels, dilations):
            blocks.append(TemporalBlock(in_ch, out_ch, kernel_size, dil, dropout))
            in_ch = out_ch
        self.network   = nn.Sequential(*blocks)
        self.fc_shared = nn.Sequential(
            nn.Linear(channels[-1], 64), nn.ReLU(), nn.Dropout(dropout),
        )
        self.head_exercise     = nn.Linear(64, N_EXERCISE_CLASSES)
        # Soft MoE: one correctness head per exercise, mixed by exercise probabilities.
        self.heads_correctness = nn.ModuleList(
            [nn.Linear(64, N_CORRECTNESS_CLASSES) for _ in range(N_EXERCISE_CLASSES)]
        )

    def forward(self, x):
        x         = x.transpose(1, 2)
        out       = self.network(x)
        out       = out[:, :, -1]
        shared    = self.fc_shared(out)
        ex_logits = self.head_exercise(shared)
        mix       = F.softmax(ex_logits, dim=1)
        cor_logits = sum(
            mix[:, i].unsqueeze(1) * head(shared)
            for i, head in enumerate(self.heads_correctness)
        )
        return ex_logits, cor_logits


print("✅ Model classes defined.")

# ── Phase 2: load checkpoints ────────────────────────────────
_TRAINING_KEYS = {"lr", "weight_decay", "correctness_weight"}

def load_dl_model(model_class, checkpoint_path):
    """Load model using the architecture config saved inside the checkpoint."""
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
    arch_cfg   = {k: v for k, v in checkpoint["model_config"].items()
                  if k not in _TRAINING_KEYS}
    model      = model_class(**arch_cfg)
    model.load_state_dict(checkpoint["model_state"])
    model.to(DEVICE)
    model.eval()
    return model, checkpoint

lstm_model, lstm_ckpt = load_dl_model(ExerciseLSTM, LSTM_PATH)
tcn_model,  tcn_ckpt  = load_dl_model(ExerciseTCN,  TCN_PATH)

print("✅ Phase 2 models loaded.")
print(f"   LSTM checkpoint — epoch {lstm_ckpt['epoch']} | score: {lstm_ckpt['score']:.4f}")
print(f"   TCN  checkpoint — epoch {tcn_ckpt['epoch']} | score: {tcn_ckpt['score']:.4f}")
print("   Both DL models normalised with the Phase 1 sklearn scaler (same pipeline as training).")


✅ Phase 1 models loaded.
   Scaler expects 26 features.
✅ Model classes defined.
✅ Phase 2 models loaded.
   LSTM checkpoint — epoch 50 | score: 0.9744
   TCN  checkpoint — epoch 27 | score: 0.9718
   Both DL models normalised with the Phase 1 sklearn scaler (same pipeline as training).


In [42]:
import warnings
warnings.filterwarnings("ignore")

# ── Phase 1 inference ─────────────────────────────────────────────────────────
def majority_vote(iterable):
    return Counter(iterable).most_common(1)[0][0]

def predict_from_window_p1(window, scaler, ex_svm, corr_svm, ex_rf, corr_rf):
    if not window:
        return None

    rows, prev = [], None
    for angles in window:
        if prev is not None:
            velocities = {f"{k}_velocity": round(abs(angles[k] - prev[k]) * 30, 2)
                          for k in angles}
        else:
            velocities = {f"{k}_velocity": 0.0 for k in angles}
        rows.append({**angles, **velocities})
        prev = angles

    df       = pd.DataFrame(rows)
    vel_cols = [c for c in df.columns if "velocity" in c]
    df[vel_cols] = df[vel_cols].clip(upper=3000)

    assert len(df.columns) == scaler.n_features_in_, (
        f"Feature mismatch: {len(df.columns)} vs {scaler.n_features_in_}"
    )
    X_scaled = scaler.transform(df)  # 26 features

    # Exercise classifiers use 26 pose features only.
    pred_ex_svm = majority_vote(ex_svm.predict(X_scaled))
    pred_ex_rf  = majority_vote(ex_rf.predict(X_scaled))

    # Correctness classifiers were trained with exercise_name_encoded as feature 27.
    # Append the exercise prediction so inference matches training feature space.
    X_cor_svm = np.hstack([X_scaled, np.full((len(X_scaled), 1), pred_ex_svm)])
    X_cor_rf  = np.hstack([X_scaled, np.full((len(X_scaled), 1), pred_ex_rf)])

    cor_preds_svm = corr_svm.predict(X_cor_svm)
    cor_preds_rf  = corr_rf.predict(X_cor_rf)

    pred_cor_svm = majority_vote(cor_preds_svm)
    pred_cor_rf  = majority_vote(cor_preds_rf)

    conf_svm = round((cor_preds_svm == 1).sum() / len(cor_preds_svm) * 100, 1)
    conf_rf  = round((cor_preds_rf  == 1).sum() / len(cor_preds_rf)  * 100, 1)

    return pred_ex_svm, pred_cor_svm, pred_ex_rf, pred_cor_rf, conf_svm, conf_rf


# ── Phase 2 inference ─────────────────────────────────────────────────────────
def predict_from_window_p2(p2_window, scaler, lstm_model, tcn_model):
    """
    Normalises raw webcam angles with the same sklearn scaler used during
    training (notebook 03 normalised the CSVs before DL training), then runs
    LSTM and TCN on a (1, MAX_SEQUENCE_LENGTH, 26) tensor.
    Returns None while the warm-up window is filling (< P2_WINDOW_SIZE frames).
    """
    if len(p2_window) < P2_WINDOW_SIZE:
        return None

    rows, prev = [], None
    for angles in p2_window:
        if prev is not None:
            velocities = {f"{k}_velocity": round(abs(angles[k] - prev[k]) * 30, 2)
                          for k in angles}
        else:
            velocities = {f"{k}_velocity": 0.0 for k in angles}
        rows.append({**angles, **velocities})
        prev = angles

    df       = pd.DataFrame(rows)
    vel_cols = [c for c in df.columns if "velocity" in c]
    df[vel_cols] = df[vel_cols].clip(upper=3000)

    X_scaled = scaler.transform(df)  # raw -> z-score, same transform as training CSVs
    PAD      = MAX_SEQUENCE_LENGTH - P2_WINDOW_SIZE
    zeros    = np.zeros((PAD, X_scaled.shape[1]), dtype=X_scaled.dtype)
    X_padded = np.vstack([X_scaled, zeros])           # real frames first, zeros at end
    tensor   = torch.tensor(X_padded, dtype=torch.float32).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        logits_ex_lstm, logits_cor_lstm = lstm_model(tensor)
        logits_ex_tcn,  logits_cor_tcn  = tcn_model(tensor)

    ex_probs_lstm  = torch.softmax(logits_ex_lstm,  dim=1)[0].cpu().numpy()
    cor_probs_lstm = torch.softmax(logits_cor_lstm, dim=1)[0].cpu().numpy()
    ex_probs_tcn   = torch.softmax(logits_ex_tcn,   dim=1)[0].cpu().numpy()
    cor_probs_tcn  = torch.softmax(logits_cor_tcn,  dim=1)[0].cpu().numpy()
    return ex_probs_lstm, cor_probs_lstm, ex_probs_tcn, cor_probs_tcn


# ── Overlay ───────────────────────────────────────────────────────────────────
def draw_overlay(frame, p1_result, p2_result, window_fill, p2_window_fill):
    def ex_name(code):
        return EXERCISE_LABELS.get(code, "?")

    def cor_label(code):
        return CORRECTNESS_LABELS.get(code, "?")

    def cor_color(code):
        return CORRECTNESS_COLORS.get(code, (200, 200, 200))

    if p1_result is not None:
        pred_ex_svm, pred_cor_svm, pred_ex_rf, pred_cor_rf, conf_svm, conf_rf = p1_result
        p1_lines = [
            # (f"Exercise  SVM : {ex_name(pred_ex_svm)}",   (255, 255, 255)),
            # (f"Form      SVM : {cor_label(pred_cor_svm)}", cor_color(pred_cor_svm)),
            # (f"Confidence SVM: {conf_svm}%",               (200, 200, 50)),
            (f"Exercise   : {ex_name(pred_ex_rf)}",    (255, 255, 255)),
            (f"Form       : {cor_label(pred_cor_rf)}",  cor_color(pred_cor_rf)),
            (f"Confidence : {conf_rf}%",                (200, 200, 50)),
        ]
    else:
        p1_lines = [("-- Phase 1: warming up... --", (150, 150, 150))]

    if p2_result is not None:
        pred_ex_lstm, pred_cor_lstm, pred_ex_tcn, pred_cor_tcn, conf_lstm, conf_tcn = p2_result
        p2_lines = [
            # (f"Exercise LSTM : {ex_name(pred_ex_lstm)}",   (255, 255, 255)),
            # (f"Form     LSTM : {cor_label(pred_cor_lstm)}", cor_color(pred_cor_lstm)),
            # (f"Confidence LSTM: {conf_lstm}%",              (200, 200, 50)),
            # (f"Exercise   : {ex_name(pred_ex_tcn)}",    (255, 255, 255)),
            # (f"Form       : {cor_label(pred_cor_tcn)}",  cor_color(pred_cor_tcn)),
            # (f"Confidence : {conf_tcn}%",               (200, 200, 50)),
        ]
    else:
        p2_lines = [("----", (150, 150, 150))]

   # status_line = (
    #    f"P1: {window_fill}/{WINDOW_SIZE}  P2: {p2_window_fill}/{P2_WINDOW_SIZE} frames",
     #   (150, 150, 150),
    #)

    all_lines = p1_lines + p2_lines# + [status_line]

    box_h   = len(all_lines) * 26 + 16
    overlay = frame.copy()
    cv2.rectangle(overlay, (8, 8), (380, box_h), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.5, frame, 0.5, 0, frame)

    for i, (text, color) in enumerate(all_lines):
        cv2.putText(frame, text, (14, 30 + i * 26),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 1, cv2.LINE_AA)

    cv2.putText(frame, "Press Q to quit",
                (14, frame.shape[0] - 12),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (150, 150, 150), 1, cv2.LINE_AA)


# ── Main webcam loop ──────────────────────────────────────────────────────────
base_options = mp_python.BaseOptions(model_asset_path=str(MODEL_PATH))
options = PoseLandmarkerOptions(
    base_options=base_options,
    running_mode=RunningMode.VIDEO,
    num_poses=1,
    min_pose_detection_confidence=0.5,
    min_pose_presence_confidence=0.5,
    min_tracking_confidence=0.5,
)

cap = cv2.VideoCapture(WEBCAM_ID)
if not cap.isOpened():
    raise RuntimeError(f"Could not open webcam (ID={WEBCAM_ID}).")

_t0 = time.time()  # reference time for VIDEO-mode timestamps

window    = deque(maxlen=WINDOW_SIZE)
p2_window = deque(maxlen=P2_WINDOW_SIZE)
last_p1   = None
last_p2   = None

# EMA state — uniform priors; updated each frame once the P2 window fills
_u4 = np.ones(N_EXERCISE_CLASSES)    / N_EXERCISE_CLASSES
_u2 = np.ones(N_CORRECTNESS_CLASSES) / N_CORRECTNESS_CLASSES
ema_ex_lstm, ema_cor_lstm = _u4.copy(), _u2.copy()
ema_ex_tcn,  ema_cor_tcn  = _u4.copy(), _u2.copy()

WINDOW_NAME = "AI Personal Trainer -- Live"
cv2.namedWindow(WINDOW_NAME, cv2.WINDOW_NORMAL)

print("Webcam inference started -- Phase 1 (RF + SVM) + Phase 2 (LSTM + TCN).")
print("   Stop: press Q in the video window, close the window, or Interrupt kernel.")

try:
    with PoseLandmarker.create_from_options(options) as landmarker:
        while True:

            if cv2.getWindowProperty(WINDOW_NAME, cv2.WND_PROP_VISIBLE) < 1:
                print("Window closed.")
                break

            ret, frame = cap.read()
            if not ret:
                print("Frame capture failed.")
                break

            h, w = frame.shape[:2]

            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image  = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
            timestamp_ms = int((time.time() - _t0) * 1000)
            result    = landmarker.detect_for_video(mp_image, timestamp_ms)

            if result.pose_landmarks and len(result.pose_landmarks) > 0:
                landmarks = result.pose_landmarks[0]
                draw_skeleton(frame, landmarks, h, w)

                angles     = extract_angles(landmarks)
                angle_vals = {k: v for k, v in angles.items()
                              if "z_diff" not in k and "rotation" not in k}
                if all(0 <= v <= 180 for v in angle_vals.values()):
                    window.append(angles)
                    p2_window.append(angles)

            if len(window) == WINDOW_SIZE:
                last_p1 = predict_from_window_p1(
                    window, scaler,
                    exercise_svm, correctness_svm,
                    exercise_rf,  correctness_rf,
                )

            if len(p2_window) == P2_WINDOW_SIZE:
                _raw = predict_from_window_p2(p2_window, scaler, lstm_model, tcn_model)
                if _raw is not None:
                    _el, _cl, _et, _ct = _raw
                    ema_ex_lstm  = EMA_ALPHA * _el + (1 - EMA_ALPHA) * ema_ex_lstm
                    ema_cor_lstm = EMA_ALPHA * _cl + (1 - EMA_ALPHA) * ema_cor_lstm
                    ema_ex_tcn   = EMA_ALPHA * _et + (1 - EMA_ALPHA) * ema_ex_tcn
                    ema_cor_tcn  = EMA_ALPHA * _ct + (1 - EMA_ALPHA) * ema_cor_tcn
                    last_p2 = (
                        int(ema_ex_lstm.argmax()),
                        int(ema_cor_lstm.argmax()),
                        int(ema_ex_tcn.argmax()),
                        int(ema_cor_tcn.argmax()),
                        round(float(ema_cor_lstm.max()) * 100, 1),
                        round(float(ema_cor_tcn.max())  * 100, 1),
                    )

            draw_overlay(frame, last_p1, last_p2,
                         window_fill=len(window), p2_window_fill=len(p2_window))

            cv2.imshow(WINDOW_NAME, frame)

            if cv2.waitKey(1) & 0xFF == ord('q'):
                print("Stopped by Q key.")
                break

except KeyboardInterrupt:
    print("Interrupted by kernel.")

finally:
    cap.release()
    cv2.destroyAllWindows()
    for _ in range(5):
        cv2.waitKey(1)
    print("Webcam released and windows closed.")


Webcam inference started -- Phase 1 (RF + SVM) + Phase 2 (LSTM + TCN).
   Stop: press Q in the video window, close the window, or Interrupt kernel.
Window closed.
Webcam released and windows closed.
